In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/sg-resale-prices/Resale flat prices based on registration date from Jan-2017 onwards.csv")

In [ ]:
df

In [ ]:

import re
import numpy as np
import pandas as pd

from joblib import dump, load

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

In [ ]:
def parse_storey_mid(x: str) -> float:
    """
    '01 TO 03' -> 2
    '10 TO 12' -> 11
    """
    if pd.isna(x):
        return np.nan
    m = re.findall(r"\d+", str(x))
    if len(m) >= 2:
        lo, hi = float(m[0]), float(m[1])
        return (lo + hi) / 2.0
    if len(m) == 1:
        return float(m[0])
    return np.nan


def parse_remaining_lease_years(x: str) -> float:
    """
    '61 years 04 months' -> 61 + 4/12
    '62 years 01 month'  -> 62 + 1/12
    """
    if pd.isna(x):
        return np.nan
    s = str(x).lower()

    years = 0
    months = 0

    m_year = re.search(r"(\d+)\s*year", s)
    if m_year:
        years = int(m_year.group(1))

    m_month = re.search(r"(\d+)\s*month", s)
    if m_month:
        months = int(m_month.group(1))

    return years + months / 12.0


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "block" in df.columns:
        df["block"] = df["block"].astype(str)

    dt = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
    df["txn_year"] = dt.dt.year.astype("float")
    df["txn_month"] = dt.dt.month.astype("float")
    df["txn_quarter"] = dt.dt.quarter.astype("float")

    df["storey_mid"] = df["storey_range"].apply(parse_storey_mid)

    df["remaining_lease_years"] = df["remaining_lease"].apply(parse_remaining_lease_years)

    df["lease_commence_year"] = pd.to_numeric(df["lease_commence_date"], errors="coerce")

    df["lease_end_year"] = df["lease_commence_year"] + 99
    df["flat_age_at_txn"] = df["txn_year"] - df["lease_commence_year"]

    return df

In [ ]:

df_fe = add_engineered_features(df)

TARGET = "resale_price"

df_fe = df_fe.dropna(subset=[TARGET]).reset_index(drop=True)

DROP_RAW = ["month", "storey_range", "remaining_lease"]

X = df_fe.drop(columns=[TARGET] + [c for c in DROP_RAW if c in df_fe.columns])
y = df_fe[TARGET].astype(float)

y_log = np.log1p(y)

X.head(), y.head()

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_log, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

categorical_cols = [c for c in ["town", "flat_type", "block", "street_name", "flat_model"] if c in X.columns]
numeric_cols = [c for c in X.columns if c not in categorical_cols]


cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
])

# Numeric pipeline
num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", cat_pipe, categorical_cols),
        ("num", num_pipe, numeric_cols),
    ],
    remainder="drop"
)

X_train_t = preprocessor.fit_transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

X_train_t.shape, X_val_t.shape, X_test_t.shape

In [ ]:
import xgboost as xgb
print(xgb.__version__)
print(xgb.build_info())  

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=5000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    objective="reg:squarederror",
    tree_method="hist",
    eval_metric="rmse",          
    early_stopping_rounds=200,   
    random_state=42,
    n_jobs=-1,

     device="cuda",
)

model.fit(
    X_train_t, y_train,
    eval_set=[(X_val_t, y_val)],
    verbose=200
)

In [ ]:

from sklearn.metrics import root_mean_squared_error


def inv_log(z):
    return np.expm1(z)

pred_val = inv_log(model.predict(X_val_t))
pred_test = inv_log(model.predict(X_test_t))

y_val_true = inv_log(y_val.values)
y_test_true = inv_log(y_test.values)

val_mae = mean_absolute_error(y_val_true, pred_val)
val_rmse  = root_mean_squared_error(y_val_true,  pred_val)

test_mae = mean_absolute_error(y_test_true, pred_test)
test_rmse = root_mean_squared_error(y_test_true, pred_test)

print("Validation:")
print(f"  MAE : {val_mae:,.0f}")
print(f"  RMSE: {val_rmse:,.0f}")
print("Test:")
print(f"  MAE : {test_mae:,.0f}")
print(f"  RMSE: {test_rmse:,.0f}")

print("\nBest iteration:", getattr(model, "best_iteration", None))
print("Best score    :", getattr(model, "best_score", None))

In [ ]:
bundle = {
    "preprocessor": preprocessor,
    "model": model,
    "feature_columns": list(X.columns),
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols
}

MODEL_PATH = "xgb_hdb_bundle.joblib"
dump(bundle, MODEL_PATH)

In [ ]:
bundle = load("xgb_hdb_bundle.joblib")

def predict_prices(bundle, df_new: pd.DataFrame) -> np.ndarray:
    df_new = add_engineered_features(df_new)

    # Drop raw columns that we dropped during training
    for col in ["month", "storey_range", "remaining_lease"]:
        if col in df_new.columns:
            df_new = df_new.drop(columns=[col])

    # Ensure all expected columns exist
    X_new = df_new.copy()
    for col in bundle["feature_columns"]:
        if col not in X_new.columns:
            X_new[col] = np.nan
    X_new = X_new[bundle["feature_columns"]]

    X_new_t = bundle["preprocessor"].transform(X_new)
    y_pred_log = bundle["model"].predict(X_new_t)
    return np.expm1(y_pred_log)

# Example row (edit as needed)
new_row = pd.DataFrame([{
    "month": "2026-01",
    "town": "ANG MO KIO",
    "flat_type": "3 ROOM",
    "block": "108",
    "street_name": "ANG MO KIO AVE 4",
    "storey_range": "04 TO 06",
    "floor_area_sqm": 67.0,
    "flat_model": "New Generation",
    "lease_commence_date": 1978,
    "remaining_lease": "50 years 06 months"
}])

pred = predict_prices(bundle, new_row)
print("Predicted resale price:", round(float(pred[0])))